# Моделирование. Проведение экспериментов. Подготовка пайплайнов обработки данных и построения модели.

# Импорты и настройки

In [1]:
# Стандартная библиотека
import os
import sys
import warnings
import joblib

# Работа с данными
import numpy as np
import pandas as pd

# ML
import lightgbm as lgb
import mlflow

# Настройки
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.options.display.float_format = "{:,.0f}".format

# Пути проекта
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.config import TRAIN_PATH, DATA_DIR, RAW_DIR

# Проверка путей
print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", os.path.exists(TRAIN_PATH))
print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("RAW_DIR exists:", os.path.exists(RAW_DIR))

PROJECT_ROOT: /home/mle_projects/bank-product-recs
TRAIN_PATH exists: True
DATA_DIR exists: True
RAW_DIR exists: True


# Загрузка данных

In [2]:
df = pd.read_csv(TRAIN_PATH, low_memory=False)
print(df.shape)
df.head()

(13647309, 48)


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,ult_fec_cli_1t,indrel_1mes,tiprel_1mes,indresi,indext,conyuemp,canal_entrada,indfall,tipodom,cod_prov,nomprov,ind_actividad_cliente,renta,segmento,ind_ahor_fin_ult1,ind_aval_fin_ult1,ind_cco_fin_ult1,ind_cder_fin_ult1,ind_cno_fin_ult1,ind_ctju_fin_ult1,ind_ctma_fin_ult1,ind_ctop_fin_ult1,ind_ctpp_fin_ult1,ind_deco_fin_ult1,ind_deme_fin_ult1,ind_dela_fin_ult1,ind_ecue_fin_ult1,ind_fond_fin_ult1,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-01-28,1375586,N,ES,H,35,2015-01-12,0,6,1,NaN,1.0,A,S,N,NaN,KHL,N,1,29,MALAGA,1,"87,218",02 - PARTICULARES,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2015-01-28,1050611,N,ES,V,23,2012-08-10,0,35,1,NaN,1,I,S,S,NaN,KHE,N,1,13,CIUDAD REAL,0,"35,549",03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2015-01-28,1050612,N,ES,V,23,2012-08-10,0,35,1,NaN,1,I,S,N,NaN,KHE,N,1,13,CIUDAD REAL,0,"122,179",03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,2015-01-28,1050613,N,ES,H,22,2012-08-10,0,35,1,NaN,1,I,S,N,NaN,KHD,N,1,50,ZARAGOZA,0,"119,776",03 - UNIVERSITARIO,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,2015-01-28,1050614,N,ES,V,23,2012-08-10,0,35,1,NaN,1,A,S,N,NaN,KHE,N,1,50,ZARAGOZA,1,NaN,03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


# Базовая предобработка

In [ ]:
df["fecha_dato"] = pd.to_datetime(df["fecha_dato"], errors="coerce")
df["fecha_alta"] = pd.to_datetime(df["fecha_alta"], errors="coerce")
df["ult_fec_cli_1t"] = pd.to_datetime(df["ult_fec_cli_1t"], errors="coerce")

product_cols = [c for c in df.columns if c.endswith("_ult1")]

for col in product_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(np.uint8)

for col in ["age", "antiguedad", "renta", "ind_nuevo", "indrel", "tipodom", "cod_prov", "ind_actividad_cliente"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Формирование таргета

In [ ]:
df = df.sort_values(["ncodpers", "fecha_dato"]).copy()

prev_products = df.groupby("ncodpers")[product_cols].shift(1)

# заменяем NaN на 0
prev_products = prev_products.fillna(0)

# таргет — только новые продукты
target_df = (df[product_cols] - prev_products).clip(lower=0)

target_cols = [f"target_{col}" for col in product_cols]
target_df.columns = target_cols

df = pd.concat([df, target_df], axis=1)

# удаляем первый месяц клиента (нет истории)
df["row_num_per_client"] = df.groupby("ncodpers").cumcount()
df = df[df["row_num_per_client"] > 0].copy()

print(df.shape)

# Feature engineering

In [ ]:
df["num_products"] = df[product_cols].sum(axis=1)
df["tenure_days"] = (df["fecha_dato"] - df["fecha_alta"]).dt.days
df["month"] = df["fecha_dato"].dt.month

#  Выбор признаков

## Числовые

In [ ]:
num_features = [
    "age",
    "antiguedad",
    "renta",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente",
    "num_products",
    "tenure_days",
    "month",
]

## Категориальные

In [ ]:
cat_features = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento",
]

### Оставим только существующие колонки:

In [ ]:
num_features = [c for c in num_features if c in df.columns]
cat_features = [c for c in cat_features if c in df.columns]
feature_cols = num_features + cat_features + product_cols

## Обработка категорий

### Для LightGBM можно закодировать категории через category.

In [ ]:
for col in cat_features:
    df[col] = df[col].fillna("UNKNOWN").astype(str).astype("category")

### Числовые:

In [ ]:
for col in num_features:
    df[col] = df[col].fillna(df[col].median())

## Time-based split

In [ ]:
train_df = df[df["fecha_dato"] < "2016-05-28"].copy()
valid_df = df[df["fecha_dato"] == "2016-05-28"].copy()

print(train_df.shape, valid_df.shape)

# Обучене модели 

## Baseline Top Popular

### Считаем самые популярные новые продукты на train:

In [ ]:
popular_targets = train_df[target_cols].sum().sort_values(ascending=False)
top_popular_products = [c.replace("target_", "") for c in popular_targets.index.tolist()]
top_popular_products[:5]

## Метрики

In [ ]:
def precision_at_k(actual, predicted, k=5):
    predicted = predicted[:k]
    if k == 0:
        return 0.0
    return len(set(actual) & set(predicted)) / k


def recall_at_k(actual, predicted, k=5):
    predicted = predicted[:k]
    if len(actual) == 0:
        return 0.0
    return len(set(actual) & set(predicted)) / len(actual)


def apk(actual, predicted, k=5):
    predicted = predicted[:k]
    score = 0.0
    hits = 0.0
    
    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            hits += 1.0
            score += hits / (i + 1.0)
    
    if len(actual) == 0:
        return 0.0
    
    return score / min(len(actual), k)


def mean_metrics(actual_list, predicted_list, k=5):
    precisions = [precision_at_k(a, p, k) for a, p in zip(actual_list, predicted_list)]
    recalls = [recall_at_k(a, p, k) for a, p in zip(actual_list, predicted_list)]
    maps = [apk(a, p, k) for a, p in zip(actual_list, predicted_list)]
    
    return {
        f"precision@{k}": float(np.mean(precisions)),
        f"recall@{k}": float(np.mean(recalls)),
        f"map@{k}": float(np.mean(maps)),
    }

## Подготовка фактических ответов для valid

In [ ]:
def extract_actual_products(row, target_cols):
    return [c.replace("target_", "") for c in target_cols if row[c] == 1]

valid_actual = valid_df[target_cols].apply(lambda row: extract_actual_products(row, target_cols), axis=1).tolist()

In [ ]:
# Baseline predictions:
K = 5
valid_pred_top_pop = [top_popular_products[:K] for _ in range(len(valid_df))]

baseline_metrics = mean_metrics(valid_actual, valid_pred_top_pop, k=K)
baseline_metrics

## Обучение LightGBM

### по одной модели на каждый продукт.

In [ ]:
models = {}
valid_scores = pd.DataFrame(index=valid_df.index)

X_train = train_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

for col in cat_features:
    X_train[col] = X_train[col].astype("category")
    X_valid[col] = X_valid[col].astype("category")

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbosity": -1,
    "seed": 42,
}

for target_col in target_cols:
    y_train = train_df[target_col].astype(int)
    y_valid = valid_df[target_col].astype(int)

    train_dataset = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features, free_raw_data=False)
    valid_dataset = lgb.Dataset(X_valid, label=y_valid, categorical_feature=cat_features, free_raw_data=False)

    model = lgb.train(
        params,
        train_dataset,
        valid_sets=[valid_dataset],
        num_boost_round=200,
        callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)]
    )

    models[target_col] = model
    valid_scores[target_col] = model.predict(X_valid, num_iteration=model.best_iteration)

## Формирование рекомендаций модели

### Важно не рекомендовать уже имеющиеся у клиента продукты.

In [ ]:
def get_model_recommendations(valid_df, score_df, product_cols, top_k=5):
    recommendations = []

    for idx in valid_df.index:
        current_products = set([p for p in product_cols if valid_df.loc[idx, p] == 1])

        scores = {
            target_col.replace("target_", ""): score_df.loc[idx, target_col]
            for target_col in score_df.columns
        }

        filtered_scores = {p: s for p, s in scores.items() if p not in current_products}
        ranked = sorted(filtered_scores.items(), key=lambda x: x[1], reverse=True)
        recs = [p for p, _ in ranked[:top_k]]
        recommendations.append(recs)

    return recommendations

In [ ]:
valid_pred_lgb = get_model_recommendations(valid_df, valid_scores, product_cols, top_k=5)
lgb_metrics = mean_metrics(valid_actual, valid_pred_lgb, k=5)
lgb_metrics

## Логирование в MLflow

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("bank_product_recommendation")

with mlflow.start_run(run_name="lightgbm_one_vs_rest_v1"):
    mlflow.log_param("model_type", "lightgbm_one_vs_rest")
    mlflow.log_param("top_k", 5)
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("num_products", len(product_cols))

    for metric_name, metric_value in baseline_metrics.items():
        mlflow.log_metric(f"baseline_{metric_name}", metric_value)

    for metric_name, metric_value in lgb_metrics.items():
        mlflow.log_metric(metric_name, metric_value)

## Сохранение модели

In [ ]:
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(models, os.path.join(MODELS_DIR, "lgbm_one_vs_rest.pkl"))

In [ ]:
for target_col, model in models.items():
    model.save_model(os.path.join(MODELS_DIR, f"{target_col}.bin"))